## Construct test cases
Set up waveform and response modules, explore mass/SNR scaling with redshift, then generate a config file and SLURM script in one step.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

from astropy.cosmology import Planck15
import astropy.units as u
from scipy.optimize import brentq

from lisaconstants import ASTRONOMICAL_YEAR
from lisaorbits import OEMOrbits
from mojito import MojitoL1File
from ruamel.yaml import YAML
from scipy.signal.windows import tukey

from src.noise import build_inv_covariance
from src.utils import inband_freqs, inner_prod_tdi, mismatch_tdi
from src.waveform import (
    ResponseConfig,
    WaveformConfig,
    build_response,
    param_names_for,
)

In [ ]:
# File path constants
ORBIT_FILE = "/data/leuven/367/vsc36785/LISA/Mojito_analysis/esa-trailing-orbits-mojito_validation_test_2.h5"
MOJITO_L1  = (
    "/scratch/leuven/367/vsc36785/MojitoLight/SIM_data/brickmarket/"
    "mojito_light_v1_0_0/data/EMRI/L1/"
    "EMRI_731d_2.5s_L1_source0_0_20251203T225446987631Z.h5"
)
NOISE_FILE = (
    "/scratch/leuven/367/vsc36785/MojitoLight/SIM_data/brickmarket/"
    "mojito_light_v1_0_0/data/NOISE/L1/"
    "NOISE_731d_2.5s_L1_source0_0_20251206T220508924302Z.h5"
)

In [ ]:
# Mojito L1 timing
with MojitoL1File(MOJITO_L1) as l1:
    ts           = l1.tdis.time_sampling
    t0_l1        = float(ts.t0)
    mojito_dt    = float(ts.dt)
    central_freq = float(l1.laser_frequency)

print(f"Mojito L1:  t0={t0_l1:.3f} s   dt={mojito_dt:.3f} s   f_laser={central_freq:.6e} Hz")

# Store raw kwargs separately — WaveformConfig may inject non-serializable
# class references into these dicts during construction, so we keep the
# originals for YAML output.
INJ_INSPIRAL_KWARGS  = {"DENSE_STEPPING": 0, "max_init_len": 1000}
INJ_SUMMATION_KWARGS = {"pad_output": True}
INJ_AMPLITUDE_KWARGS = {}

# Injection waveform config
inj_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=5.0,
    T=2.0,
    evolve_chi1=True,
    include_1PA_amps=True,
    inspiral_kwargs=INJ_INSPIRAL_KWARGS,
    summation_kwargs=INJ_SUMMATION_KWARGS,
    amplitude_kwargs=INJ_AMPLITUDE_KWARGS,
)
print(f"Injection : {inj_wcfg.model}  evolve_chi1={inj_wcfg.evolve_chi1}  1PA_amps={inj_wcfg.include_1PA_amps}")

In [ ]:
REC_INSPIRAL_KWARGS  = {"DENSE_STEPPING": 0, "max_init_len": 1000}
REC_SUMMATION_KWARGS = {"pad_output": True}
REC_AMPLITUDE_KWARGS = {}

# Recovery waveform config
rec_wcfg = WaveformConfig(
    model="1PAT1R",
    dt=5.0,
    T=2.0,
    evolve_chi1=True,
    include_1PA_amps=True,
    inspiral_kwargs=REC_INSPIRAL_KWARGS,
    summation_kwargs=REC_SUMMATION_KWARGS,
    amplitude_kwargs=REC_AMPLITUDE_KWARGS,
)
print(f"Recovery  : {rec_wcfg.model}  evolve_chi1={rec_wcfg.evolve_chi1}  1PA_amps={rec_wcfg.include_1PA_amps}")
if inj_wcfg.model != rec_wcfg.model:
    print(" Models differ — systematic bias is expected.")

In [ ]:
# Shared response config (same for injection and recovery)
resp_cfg = ResponseConfig(
    orbit_file=ORBIT_FILE,
    tdi_gen="2nd generation",
    tdi_chan="XYZ",
    order=40,
    offset=550.0,
    n_samples_delay=1000,
    t_buffer=10000.0,
    flip_hx=True,
    is_ecliptic_latitude=False,
)
print(f"Response  : {resp_cfg.tdi_chan}  {resp_cfg.tdi_gen}  order={resp_cfg.order}")

In [ ]:
# Derived timing (mirrors PE_response.py)
DT         = inj_wcfg.dt
oem_orbits = OEMOrbits.from_included("esa-trailing")
t0_orbits  = float(oem_orbits.t_start) + 10.0
T_response = (
    inj_wcfg.T
    + (2 * resp_cfg.offset + 2 * resp_cfg.n_samples_delay * DT) / ASTRONOMICAL_YEAR
)
t0_l0  = t0_l1 - resp_cfg.n_samples_delay * mojito_dt
t_init = t0_l0 - resp_cfg.offset

print(f"t_init     = {t_init:.3f} s")
print(f"T_response = {T_response:.6f} yr")

print("\nBuilding injection response ...")
inj_response = build_response(inj_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=True)

print("Building recovery response ...")
rec_response = build_response(rec_wcfg, resp_cfg, t_init, t0_orbits, T_response, use_gpu=True)

print("Done.")

## Cosmology and mass scaling

Helper functions for computing the luminosity distance (Planck15), visualising how source-frame masses and the SNR proxy scale with redshift, and finding the redshift that yields a desired SNR for fixed detector-frame masses.

Key relations:
- $m_{\rm det} = m_{\rm src}\,(1+z)$
- $d_L = d_L(z)$ from Planck15 cosmology
- ${\rm SNR} \propto \mu_{\rm src}/d_L = \mu_{\rm det}/[(1+z)\,d_L(z)]$

In [ ]:
def get_D_at_z(z):
    """Luminosity distance at redshift z under Planck15 cosmology [Gpc]."""
    return float(Planck15.luminosity_distance(z).to(u.Gpc).value)


def source_params_from_detector(d_L, M_det, mu_det):
    """
    Given a luminosity distance and detector-frame masses, return the
    redshift and source-frame masses to place in the YAML config.

    The config stores source-frame masses; PE_response.py redshifts them
    internally via  m_det = m_src * (1 + z).  The SNR scales as
        SNR ∝ mu_src / d_L = mu_det / [(1+z) · d_L]
    so choosing d_L controls the SNR for fixed detector-frame masses.

    Parameters
    ----------
    d_L    : luminosity distance [Gpc]
    M_det  : detector-frame primary mass [M_sun]
    mu_det : detector-frame secondary mass [M_sun]

    Returns
    -------
    z      : redshift consistent with d_L under Planck15
    M_src  : source-frame primary mass [M_sun]   = M_det  / (1 + z)
    mu_src : source-frame secondary mass [M_sun] = mu_det / (1 + z)
    """
    z = brentq(lambda zz: get_D_at_z(zz) - d_L, 1e-4, 20.0, xtol=1e-6)
    return z, M_det / (1.0 + z), mu_det / (1.0 + z)

In [ ]:
# Illustrate mass and distance scaling for a representative detector-frame mass pair.
# Edit M_det_ex / mu_det_ex to match your planned source.
M_det_ex  = 1e6    # primary,   detector frame [M_sun]
mu_det_ex = 10.0   # secondary, detector frame [M_sun]

z_arr       = np.linspace(0.01, 5.0, 500)
dL_arr      = np.array([get_D_at_z(z) for z in z_arr])
M_src_arr   = M_det_ex  / (1.0 + z_arr)
mu_src_arr  = mu_det_ex / (1.0 + z_arr)
snr_proxy   = 1.0 / ((1.0 + z_arr) * dL_arr)   # SNR ∝ mu_src/d_L (normalised)
snr_proxy  /= snr_proxy.max()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(z_arr, M_src_arr / 1e6, label=r"$M_{\rm src}\;[10^6\,M_\odot]$")
axes[0].plot(z_arr, mu_src_arr,       label=r"$\mu_{\rm src}\;[M_\odot]$")
axes[0].set_xlabel("z")
axes[0].set_title(
    f"Source-frame masses\n"
    rf"($M_{{\rm det}}={M_det_ex:.0e}$, $\mu_{{\rm det}}={mu_det_ex}$)"
)
axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].semilogy(z_arr, dL_arr)
axes[1].set_xlabel("z"); axes[1].set_ylabel("$d_L$ [Gpc]")
axes[1].set_title("Luminosity distance (Planck15)")
axes[1].grid(alpha=0.4, which="both")

axes[2].plot(z_arr, snr_proxy)
axes[2].set_xlabel("z"); axes[2].set_ylabel("SNR proxy (normalised)")
axes[2].set_title(r"SNR $\propto \mu_{\rm src}/d_L$ (fixed $M_{\rm det},\,\mu_{\rm det}$)")
axes[2].grid(alpha=0.4)

fig.tight_layout(); plt.show()

## Mismatch and SNR utilities

Compute SNR and mismatch between injection and recovery using the noise-weighted inner product.

In [ ]:
def snr_tdi(h_fft: cp.ndarray, inv_cov: cp.ndarray) -> float:
    return float(cp.sqrt(inner_prod_tdi(h_fft, h_fft, inv_cov)))


def _to_vector(params: dict, model: str) -> list:
    """Convert a params dict to the ordered vector expected by the waveform model."""
    z   = float(params.get("z", 0.0))
    vec = []
    for n in param_names_for(model):
        val = float(params[n])
        if n in ("M", "mu"):
            val *= (1.0 + z)
        vec.append(val)
    return vec

In [ ]:
# Nominal grid for inverse covariance (rebuilt automatically if N_t differs).
N_t_nominal = int(round(T_response * ASTRONOMICAL_YEAR / DT))
freqs_inband_nom, mask_nom = inband_freqs(N_t_nominal, DT, filter_freq=True)

print(f"N_t_nominal = {N_t_nominal}   n_inband = {int(mask_nom.sum())}")
print("Building inverse covariance ...")

inv_cov, psd_diag = build_inv_covariance(
    NOISE_FILE, central_freq,
    cp.asnumpy(freqs_inband_nom), DT, N_t_nominal,
    channels=resp_cfg.tdi_chan,
)
print(f"inv_cov shape: {inv_cov.shape}")

In [ ]:
# ── Masses and distance ───────────────────────────────────────────────────────
# Specify detector-frame masses and luminosity distance.
# source_params_from_detector() inverts Planck15 d_L(z) to get z, then
# returns the source-frame masses that go into the YAML config.
M_det  = 2e6    # detector-frame primary mass [M_sun]
mu_det = 200.0  # detector-frame secondary mass [M_sun]
d_L    = 3.0    # luminosity distance [Gpc]

z, M, mu = source_params_from_detector(d_L, M_det, mu_det)
print(f"d_L = {d_L} Gpc  →  z = {z:.4f}")
print(f"M_src  = {M:.4e} M_sun   (M_det  = {M_det:.4e})")
print(f"mu_src = {mu:.4e} M_sun   (mu_det = {mu_det:.4e})")

# ── Remaining intrinsic parameters ───────────────────────────────────────────
a    = 0.0002   # primary spin [-0.999 … 0.999]
p0   = 15.7905  # initial semi-latus rectum [M]
e0   = 0.0      # initial eccentricity
chi2 = 0.9      # secondary spin (1PAT1R only)

# Extrinsic / angular parameters (fix seed for reproducibility)
rng = np.random.default_rng(seed=42)
theta_S    = rng.uniform(0.0, np.pi)
phi_S      = rng.uniform(0.0, 2 * np.pi)
theta_K    = rng.uniform(0.0, np.pi)
phi_K      = rng.uniform(0.0, 2 * np.pi)
Phi_phi0   = rng.uniform(0.0, 2 * np.pi)
Phi_theta0 = rng.uniform(0.0, 2 * np.pi)
Phi_r0     = rng.uniform(0.0, 2 * np.pi)

inj_params = dict(
    M=M, mu=mu, a=a, p0=p0, e0=e0, chi2=chi2, x_I0=1.0, d_L=d_L,
    theta_S=theta_S, phi_S=phi_S,
    theta_K=theta_K, phi_K=phi_K,
    Phi_phi0=Phi_phi0, Phi_theta0=Phi_theta0, Phi_r0=Phi_r0,
    z=z,
)

print("\nInjection params (source frame):")
for k, v in inj_params.items():
    print(f"  {k:12s} = {v:.6g}")

In [ ]:
# Recovery params: same physical source in the recovery model's parameter space.
# Edit here to probe systematics (e.g. different evolve_chi1 / include_1PA_amps).
rec_params = dict(inj_params)

In [ ]:
windowing = True

inj_vec = _to_vector(inj_params, inj_wcfg.model)
rec_vec = _to_vector(rec_params, rec_wcfg.model)

# Generate injection TDI
print("Generating injection TDI ...")
xyz_inj = inj_response(*inj_vec)
N_t     = xyz_inj.shape[1]
window  = cp.asarray(tukey(N_t, alpha=0.01)) if windowing else cp.ones(N_t)
freqs_inband, mask = inband_freqs(N_t, DT, filter_freq=True)
xyz_inj_fft = cp.fft.rfft(xyz_inj * window, axis=1)[:, mask]
print(f"  N_t = {N_t}   n_inband = {int(mask.sum())}")

# Rebuild inv_cov on the exact grid if N_t differed from the nominal estimate
if N_t != N_t_nominal:
    print(f"  N_t differs from nominal ({N_t_nominal}) — rebuilding inv_cov ...")
    inv_cov, psd_diag = build_inv_covariance(
        NOISE_FILE, central_freq,
        cp.asnumpy(freqs_inband), DT, N_t,
        channels=resp_cfg.tdi_chan,
    )

In [ ]:
# Generate recovery TDI at injection params and compute diagnostics
print("Generating recovery TDI at injection params ...")
xyz_rec     = rec_response(*rec_vec)
xyz_rec_fft = cp.fft.rfft(xyz_rec * window, axis=1)[:, mask]

snr_inj = snr_tdi(xyz_inj_fft, inv_cov)
mm      = mismatch_tdi(xyz_inj_fft, xyz_rec_fft, inv_cov)
snr_res = snr_tdi(xyz_inj_fft - xyz_rec_fft, inv_cov)

print(f"\nInjection SNR           : {snr_inj:.2f}")
print(f"Mismatch (inj vs rec)   : {mm:.3e}")
print(f"SNR of residual         : {snr_res:.2f}")

if snr_inj < 20:
    print("WARNING: SNR below 20 — source may not be recoverable.")

In [ ]:
# ── Optional: adjust d_L to hit a desired SNR ────────────────────────────────
# SNR ∝ mu_det / [(1+z) · d_L].  For fixed detector-frame masses the ratio
# (1+z)·d_L is a monotone function of d_L (via Planck15), so:
#
#   d_L_new ≈ d_L_ref * snr_inj / target_snr   (exact only at low z,
#                                                 use as a starting guess)
#
# Then call source_params_from_detector with the new d_L to get the exact
# z and source-frame masses.  Uncomment and adapt:

# target_snr = 50.0
# d_L_new    = d_L * snr_inj / target_snr          # approximate starting guess
# z_new, M_new, mu_new = source_params_from_detector(d_L_new, M_det, mu_det)
# print(f"target SNR ≈ {target_snr}")
# print(f"  d_L_new = {d_L_new:.3f} Gpc   z_new = {z_new:.4f}")
# print(f"  M_src   = {M_new:.4e} M_sun   mu_src = {mu_new:.4e} M_sun")
# # Update d_L, M_det, mu_det in the inj_params cell above and rerun.

## Generate config and SLURM script

Set `RUN_NAME` and `RUNTIME` in the cell below. The name is propagated automatically to:
- `Sampler/name` inside the YAML config
- `--job-name` and the filename of the SLURM script

Injection parameters are taken from `inj_params` defined above.

In [ ]:
# ─── Specify these before generating ─────────────────────────────────────────
RUN_NAME  = "my_run_name"   # → config/{RUN_NAME}.yaml  and  scripts/submit_{RUN_NAME}.sh
RUNTIME   = "15:00:00"      # SLURM wall-time  (hh:mm:ss)

# Sampler / fixed-parameter settings
FIXED_PARAMS = ["x_I0", "e0", "Phi_theta0", "Phi_r0"]
N_TEMPS      = 2
N_WALKERS    = 40
NUM_SAMPLES  = 10000
D_SCALE      = 5    # prior half-width scale (see PE_response.py)

In [ ]:
def _wf_block(wcfg, inspiral_kwargs, summation_kwargs, amplitude_kwargs):
    """Build a YAML-serializable waveform block from a WaveformConfig.

    The raw *_kwargs dicts are passed explicitly because WaveformConfig may
    inject non-serializable class references into the stored copies.
    """
    block = {
        "model":                    wcfg.model,
        "evolve_chi1":              wcfg.evolve_chi1,
        "include_1PA_amps":         wcfg.include_1PA_amps,
        "dt":                       wcfg.dt,
        "T":                        wcfg.T,
        "mode_selection_threshold": wcfg.mode_selection_threshold,
        "inspiral_kwargs":          dict(inspiral_kwargs),
        "summation_kwargs":         dict(summation_kwargs),
        "amplitude_kwargs":         dict(amplitude_kwargs),
    }
    if wcfg.lmax is not None:
        block["lmax"] = wcfg.lmax
    return block


cfg_out = {
    "Data": {
        "orbit_file":     ORBIT_FILE,
        "mojito_l1_file": MOJITO_L1,
        "noise_file":     NOISE_FILE,
    },
    "Injection": {
        "EMRI":     {k: float(v) for k, v in inj_params.items()},
        "Waveform": _wf_block(inj_wcfg, INJ_INSPIRAL_KWARGS, INJ_SUMMATION_KWARGS, INJ_AMPLITUDE_KWARGS),
    },
    "Response": {
        "tdi_gen":              resp_cfg.tdi_gen,
        "tdi_chan":             resp_cfg.tdi_chan,
        "order":                resp_cfg.order,
        "offset":               resp_cfg.offset,
        "n_samples_delay":      resp_cfg.n_samples_delay,
        "t_buffer":             resp_cfg.t_buffer,
        "flip_hx":              resp_cfg.flip_hx,
        "is_ecliptic_latitude": resp_cfg.is_ecliptic_latitude,
    },
    "Recovery": {
        "Waveform": _wf_block(rec_wcfg, REC_INSPIRAL_KWARGS, REC_SUMMATION_KWARGS, REC_AMPLITUDE_KWARGS),
    },
    "Sampler": {
        "name":               RUN_NAME,
        "windowing":          True,
        "filter_freq":        True,
        "num_samples":        NUM_SAMPLES,
        "burn_in":            0,
        "use_gpu":            True,
        "n_temps":            N_TEMPS,
        "n_walkers":          N_WALKERS,
        "d":                  D_SCALE,
        "continue_run":       False,
        "sampling_data_path": "./sampling_data",
        "plots_path":         "./Plots",
        "fixed_params":       FIXED_PARAMS,
    },
}

In [ ]:
import os

CONFIG_DIR = "config"
SCRIPT_DIR = "scripts"
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(SCRIPT_DIR, exist_ok=True)

# ── Write YAML config ─────────────────────────────────────────────────────────
config_fname = f"{RUN_NAME}.yaml"
config_path  = os.path.join(CONFIG_DIR, config_fname)

yaml_obj = YAML()
yaml_obj.default_flow_style = False
yaml_obj.indent(mapping=2, sequence=4, offset=2)
with open(config_path, "w") as fh:
    yaml_obj.dump(cfg_out, fh)

# ── Write SLURM script ────────────────────────────────────────────────────────
slurm_fname = f"submit_{RUN_NAME}.sh"
slurm_path  = os.path.join(SCRIPT_DIR, slurm_fname)
HOME_VSC    = "/data/leuven/367/vsc36785/LISA/Few_1PAT1R_Scripts/validation/PE_test_runs"

slurm_lines = [
    "#!/bin/bash -l",
    f"#SBATCH -t {RUNTIME}",
    "#SBATCH --cluster=wice",
    "#SBATCH --nodes=1",
    "#SBATCH --ntasks=18",
    "#SBATCH --partition=gpu_a100",
    "#SBATCH --gpus-per-node=1",
    "#SBATCH --output=./out/%x_%j_%a.out",
    "#SBATCH --mail-type=FAIL,BEGIN,END",
    "#SBATCH --mail-user=bert.depoorter@student.kuleuven.be",
    "#SBATCH -A lp_lisagw",
    f"#SBATCH --job-name={RUN_NAME}",
    "",
    "module load GCC/11.3.0 GSL CUDA/12 FFTW/3.3.10-GCC-11.3.0",
    "conda activate emri_env_ddpc",
    "nvidia-smi",
    "",
    f"HOME_FOLDER={HOME_VSC}",
    "SAMPLE_SCRIPTS=$HOME_FOLDER/PE_response.py",
    f"INFERENCE_PARAMS=$HOME_FOLDER/config/{config_fname}",
    "",
    "python $SAMPLE_SCRIPTS --config=$INFERENCE_PARAMS",
]
with open(slurm_path, "w") as fh:
    fh.write("\n".join(slurm_lines) + "\n")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Config  → {config_path}")
print(f"SLURM   → {slurm_path}")
print(f"  name       : {RUN_NAME}")
print(f"  runtime    : {RUNTIME}")
print(f"  SNR (inj)  : {snr_inj:.2f}")
print(f"  mismatch   : {mm:.3e}")
print(f"  fixed      : {FIXED_PARAMS}")